# 4. Forced response: figures

The figures for the forced response, drawn from the results notebook 03 saved, so nothing slow runs here. Each section matches a section of notebook 03, where the method is explained.

1. Quantile change relative to hist-nat
2. Change in width (the hist-nat bootstrap)
3. Quantile response: Q05, Q50, Q95
4. Mean change (t-test and member-block test)
5. Signal to noise
6. Share of the historical change from each forcing
7. Additivity
8. Mean and variability change: where both have changed
9. Zonal structure and the sea-ice edge (after Bracegirdle et al., 2024)

Most figures come in two forms: a **multi-model summary** (the median across models, hatched where the models do not agree) and a **single-model detail** for `PLOT_SEL['model']`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..', '../../extant-functions']   # this repository (the extant package) and plotting_modules

import logging
import warnings
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from dask.distributed import wait
from plotting_modules import formatting, maps
from extant import config, loading, paths, preprocessing as pp, storage
from extant.logger import setup_logging
from extant.plots import methods

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Dask warns about every large task graph; the graphs here are large on purpose
warnings.filterwarnings('ignore', message='.*Sending large graph of size.*', category=UserWarning)

from plotting_modules import figures, timeseries, utils
from plotting_modules.maps import plot_hatch, polar_grid
from extant import response_change as rc, sea_ice, significance as sig, zonal
from extant.plots import multimodel as mm, response_change as rcp, sea_ice as sip, zonal as zp

## Settings

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]
UNITS = var.units

WINDOW = config.WINDOW
LAST_FULL_YEAR = config.LAST_FULL_YEAR
PLOT_SEL = config.PLOT_SEL
MAIN_EXPERIMENTS = config.MAIN_EXPERIMENTS
FORCED_EXPERIMENTS = config.FORCED_EXPERIMENTS
SIGN_THRESHOLD = config.SIGN_THRESHOLD
SIGNIFICANT_THRESHOLD = config.SIGNIFICANT_THRESHOLD

#(t): Significance level of the width and mean tests, and |S/N| counted as emerged
ALPHA = 0.05
SN_THRESHOLD = 2


def result_path(name):
    """Where notebook 03 saved a result: results/<variable>/<run>/<name>.zarr."""
    return paths.result(name, VARIABLE, RUN)

### Colour scales

In [ ]:
#(t): Percent of historical: white near 0 and 100%, purple and red beyond
frac_bounds = np.arange(-3, 3.25, .25)
my_cmap, my_norm = utils.make_diverging_colormap(np.arange(-200, 225, 25), purple_start=-100, red_start=100)

#(t): S/N: white for |S/N| < 2
sn_bounds = np.arange(-3, 3.25, .25)
sn_cmap, sn_norm = utils.make_diverging_colormap(sn_bounds, -2, 2)

## Open the data and the saved results

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (used only for the figures at one grid point) and `RUN`, which says which saved results to open.

### Sample data

In [ ]:
lesfmip_season_tree = pp.seasonal_tree(loading.open_lesfmip_sample(experiments=config.MAIN_EXPERIMENTS), VARIABLE).compute()
RUN = 'sample'

### Full data (JASMIN)

In [ ]:
lesfmip_season_tree = loading.open_seasonal(VARIABLE, experiments=config.MAIN_EXPERIMENTS)
RUN = 'full'

### The saved results

Everything notebook 03 computed, loaded into memory (a few GB at most for the full data).

In [ ]:
q_histnat_base_da = storage.open_dataset(result_path('q_histnat_base'), load=True)
lesfmip_smooth_q_da = storage.open_dataset(result_path('smooth_quantiles'), load=True)
qrange_change_ds = storage.open_dataset(result_path('qrange_change'), load=True)
quantile_response_da = storage.open_dataarray(result_path('quantile_response'), load=True)
antarctic_quantile_curve_da = storage.open_dataarray(result_path('quantile_curve'), load=True)
ttest_ds = storage.open_dataset(result_path('ttest'), load=True)
mean_block_ds = storage.open_dataset(result_path('mean_block'), load=True)
lesfmip_seasonal_sn_da = storage.open_dataset(result_path('sn'), load=True)
own_changes_da = storage.open_dataarray(result_path('own_changes'), load=True)
mean_response_da = storage.open_dataarray(result_path('mean_response'), load=True)
tails_ds = storage.open_dataset(result_path('tails'), load=True)
zonal_mean_ds = storage.open_dataset(result_path('zonal_mean_test'), load=True)
zonal_width_ds = storage.open_dataset(result_path('zonal_width_test'), load=True)

### Sea ice

The sea-ice edge on the maps and zonal-mean figures comes from notebook 01, section 5. Run **one** of the next two sections. Both leave `ice_extents` (each member's seasonal extent), `ice_climatology` (the ensemble-mean concentration over the final years) and `ice_edges` (the equivalent latitude of the edge in the final years); without sea ice they are `None` and the figures leave the edge out.

#### Without sea ice

For the sample (which has none), or before notebook 01 has saved it.

In [ ]:
ice_extents = ice_climatology = ice_edges = None

#### With sea ice (JASMIN)

In [ ]:
ice_extents = storage.open_tree(paths.result('extent', 'siconc', 'full'), load=True)
ice_climatology = storage.open_dataarray(paths.result('climatology', 'siconc', 'full'), load=True)
ice_edges = sea_ice.edge_latitude(ice_extents, years=WINDOW)

## 1. Quantile change relative to hist-nat

$\Delta Q_p(t) = \tilde{Q}_p^{\,\mathrm{exp}}(t) - Q_p^{\,\mathrm{nat}}$: the smoothed rolling quantiles of each experiment minus hist-nat's whole-record quantiles (notebook 03, section 1). The maps show the last year whose 21-year window is complete, i.e. the centre of the final 21 years (2003).

In [ ]:
lesfmip_change_quantile_da = lesfmip_smooth_q_da - q_histnat_base_da

#(t): The last year whose WINDOW-year window is complete
final_change_quantile = lesfmip_change_quantile_da.isel(year=LAST_FULL_YEAR)

### Maps

**Change in width, every model.** $\Delta(Q_{90} - Q_{10})$ in DJF, experiments down and models across. Red is a wider distribution than hist-nat's, blue a narrower one.

In [ ]:
fig = maps.polar_grid(
    (final_change_quantile.sel(quantile=0.9) - final_change_quantile.sel(quantile=0.1)).sel(experiment=MAIN_EXPERIMENTS)[VARIABLE],
    row_dim='experiment',
    col_dim='model',
    sel={'season': 'DJF'},#, 'quantile': 0.5},
    levels=np.arange(-1, 1.1, .1),
    cmap='RdBu_r',
    cbar_label='p90 - p10 ' + f'({UNITS})',
)

**Change in the median, every model.** $\Delta Q_{50}$ in DJF, the same layout: the shift of the middle of the distribution.

In [ ]:
fig = maps.polar_grid(
    final_change_quantile.sel(experiment=MAIN_EXPERIMENTS)[VARIABLE],
    row_dim='experiment',
    col_dim='model',
    sel={'season': 'DJF', 'quantile': 0.5},
    levels=np.arange(-5, 5.5, .5) / 2,
    cmap='RdBu_r',
    cbar_label='Anomaly ' + f'({UNITS})',
)

**Low, middle and high quantile together.** For historical in DJF, one row per model: $\Delta Q_{10}$, $\Delta Q_{50}$, $\Delta Q_{90}$ and the change in width between them. Equal colours across a row mean a pure shift.

In [ ]:
panels = figures.quantile_matrix(
    ds=final_change_quantile.sel(experiment='historical').isel(season=0)[VARIABLE],#,.sel(season=['DJF']),
    row_dim='model',
    quant_dim='quantile',
    q_low=0.1,
    q_high=0.9,
    raw_levels=np.arange(-5, 5.5, .5)/2,
    diff_levels=np.linspace(-2, 2, 9)/2,
    cmap='RdBu_r',
    title=None,
    raw_label = 'Anomaly ' + f'({UNITS})',
    diff_label = 'Difference ' + f'({UNITS})'
)

**Every quantile and season, one model.** `PLOT_SEL`'s change in each of the seven quantiles (rows) in each season (columns).

In [ ]:
#(t): Every quantile and season, one model and experiment
panels = maps.polar_grid(
    final_change_quantile.sel(**PLOT_SEL)[VARIABLE],
    row_dim='quantile', col_dim='season', levels=np.linspace(-3, 3, 13), cmap='RdBu_r',
    label_fmt={'quantile': lambda q: f'p{q * 100:g}'},
    title=f"{PLOT_SEL['model']} {PLOT_SEL['experiment']}: change in each quantile",
    cbar_label=f'Change ({UNITS})', tag=False,
)

### Smoothed quantiles at a point

Each season's smoothed quantiles at `config.POINT` (bottom row) and the width between each symmetric pair (top row).

In [ ]:
panels = figures.quantile_summary(
    lesfmip_smooth_q_da.sel(**PLOT_SEL)[VARIABLE].sel(lat=config.POINT['lat'], lon=config.POINT['lon'], method='nearest'),
    q_pairs=((0.01, 0.99), (0.1, 0.9), (0.05, 0.95)), q_med=0.5,
)

### Model agreement

How many models have the distribution widening, and how many narrowing, for each quantile pair: $\Delta(Q_{1-p} - Q_p)$ for $p$ = 0.1, 0.05, 0.01.

In [ ]:
#(t): Each pair is labelled by its upper quantile: (0.1, 0.9) is 0.9
quantile_pairs = [(0.1, 0.9), (0.05, 0.95), (0.01, 0.99)]
final_quantile_spread = xr.concat(
    [(final_change_quantile.sel(quantile=upper) - final_change_quantile.sel(quantile=lower)).assign_coords(quantile=upper)
     for lower, upper in quantile_pairs],
    dim='quantile')[VARIABLE]

models_spread_sign_count = xr.concat(
    [(final_quantile_spread > 0).sum('model'), (final_quantile_spread < 0).sum('model')],
    dim=xr.IndexVariable('change', ['increase', 'decrease']),
)

In [ ]:
n_models = final_quantile_spread.sizes['model']
model_cmap, model_norm = rcp.count_colormap(n_models, base_cmap='winter')

The number of models with a wider (top row) and a narrower (bottom row) 1–99% range in DJF, one column per experiment.

In [ ]:
panels = maps.polar_grid(
    models_spread_sign_count.sel({'season': 'DJF', 'quantile': 0.99}),
    col_dim='experiment',
    row_dim='change',
    levels=np.arange(-0.5, n_models + 1.5),
    cmap=model_cmap,
    norm=model_norm,
    discrete=True,
    cbar_label='Number of models',
    cbar_height=0.33,
)

### Multi-model summary

The median across models, with hatching where fewer than 80% of the models agree on the sign (there is no significance test for these two).

In [ ]:
#(t): Median warming and the Q95−Q05 spread at the final valid year; no per-model test, so hatching is sign agreement only
qw_fields = {
    'ΔQ50': final_change_quantile[VARIABLE].sel(quantile=0.5, experiment=FORCED_EXPERIMENTS),
    'Δ(Q95 − Q05)': final_quantile_spread.sel(quantile=0.95, experiment=FORCED_EXPERIMENTS),
}

for name, field in qw_fields.items():
    mm.model_agreement_maps(
        sig.model_agreement(field, sign_threshold=SIGN_THRESHOLD),
        title=f'{name} relative to hist-nat: multi-model median',
        cbar_label=name + f' ({UNITS})',
    )

The same two quantities as Antarctic (area-weighted) means: one cell per model and experiment, one panel per season, with the multi-model median at the bottom.

In [ ]:
for name, field in qw_fields.items():
    mm.model_heatmap(
        sig.area_mean(field).compute(),
        title=f'Antarctic-mean {name} relative to hist-nat',
        cbar_label=name + f' ({UNITS})',
    )

## 2. Change in width: the hist-nat bootstrap

$\Delta W = W_{\mathrm{exp}} - W_{\mathrm{nat}}$, the final 21 years' Q95 − Q05 against hist-nat's whole record, tested against 10-member, 21-year hist-nat trials (notebook 03, section 2, where the test is drawn step by step).

### Multi-model summary

One figure per metric: rows are experiments, columns are seasons, colour is the multi-model median.
Hatching follows IPCC AR6: `////` fewer than `SIGNIFICANT_THRESHOLD` of models are significant (no robust signal);
`xxxx` enough models are significant but fewer than `SIGN_THRESHOLD` agree on the sign (conflicting).
Without a significance test, `////` marks fewer than `SIGN_THRESHOLD` of models agreeing on the sign.

In [ ]:
#(t): Collapse the model dim: the multi-model median change and how many models find it significant
qrange_agreement = sig.model_agreement(
    qrange_change_ds['qrange_change'], qrange_change_ds['qrange_significant'],
    sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD,
)

In [ ]:
mm.model_agreement_maps(qrange_agreement, title=f'Change in Q95 − Q05 (final {WINDOW} years − hist-nat full record)',
                     cbar_label='Δ(Q95 − Q05)')

Antarctic means per model; the number in each cell is the share of the area where that model's change is significant.

In [ ]:
#(t): Overview: Antarctic-mean change per model; text = % of the area where that model is significant
mm.model_heatmap(
    sig.area_mean(qrange_change_ds['qrange_change']).compute(),
    title=f'Antarctic-mean change in Q95 − Q05 (final {WINDOW} years − hist-nat full record)',
    cbar_label='Δ(Q95 − Q05)',
    annot=sig.area_mean(
        qrange_change_ds['qrange_significant'].where(qrange_change_ds['qrange_change'].notnull())).compute(),
)

### Single-model detail (`PLOT_SEL`)

Three figures for one season: the two widths being compared (hist-nat's whole record, the experiment's final 21 years); the change with the 2.5th and 97.5th percentiles of the hist-nat trials, so the change is significant wherever it is outside them; and the p-value.

In [ ]:
from plotting_modules.maps import polar_grid

season = "MAM"
ds = qrange_change_ds.sel(**PLOT_SEL, season=season)


def panel_fields(names, labels):
    return xr.concat([ds[name] for name in names], dim=xr.IndexVariable("field", labels))


# ============================================================
# 1. Q95-Q05 RANGES
# ============================================================

ranges = panel_fields(
    ["hist_nat_qrange", "experiment_qrange"],
    ["Hist-nat — full record", f"{PLOT_SEL['experiment']} — final {WINDOW} years"],
)
range_max = float(ranges.quantile(0.98))
range_panels = polar_grid(
    ranges, col_dim="field", levels=np.linspace(0, range_max, 13), cmap="viridis",
    title=f"Q95 − Q05 range — {season}", cbar_label=f"Q95 − Q05 ({UNITS})", tag=False,
)


# ============================================================
# 2. THE CHANGE AND THE HIST-NAT BOOTSTRAP RANGE
# ============================================================

sampling = panel_fields(
    ["qrange_change", "null_lower", "null_upper"],
    ["Change", "hist-nat trials: 2.5th percentile", "hist-nat trials: 97.5th percentile"],
)
sampling_max = float(np.abs(sampling).quantile(0.98))
sampling_panels = polar_grid(
    sampling, col_dim="field", levels=np.linspace(-sampling_max, sampling_max, 13), cmap="RdBu_r",
    title=f"Change in Q95 − Q05 and the {sig.N_BOOTSTRAP_MEMBERS}-member hist-nat range — {season}",
    cbar_label=f"Δ(Q95 − Q05) ({UNITS})", tag=False,
)


# ============================================================
# 3. BOOTSTRAP P-VALUE
# ============================================================

pvalue_panels = polar_grid(
    ds["qrange_pvalue"].expand_dims(field=["hist-nat bootstrap p-value"]), col_dim="field",
    levels=np.linspace(0, 0.10, 11), cmap="magma_r",
    title=f"{PLOT_SEL['experiment']} vs the hist-nat bootstrap — {season}",
    cbar_label="p-value (values > 0.10 saturated)", tag=False,
)

The change in width in every season, hatched where it is inside the range of the hist-nat trials (not significant).

In [ ]:
from matplotlib.patches import Patch

from plotting_modules.maps import polar_grid, plot_hatch

plot_ds = qrange_change_ds.sel(**PLOT_SEL)
change = plot_ds["qrange_change"]

# One shared, zero-centred scale across all seasons.
limit = float(np.nanquantile(np.abs(change.values), 0.98))
levels = np.linspace(-limit, limit, 13)

panels = polar_grid(
    change,
    col_dim="season",
    levels=levels,
    cmap="RdBu_r",
    title=f"Change in Q95 − Q05: final {WINDOW} years − hist-nat full record",
    cbar_label=f"Δ(Q95 − Q05) ({UNITS})",
    tag=False,
)

#(t): Hatched where the change is inside the hist-nat bootstrap range (not significant)
not_significant = plot_ds["qrange_pvalue"].notnull() & ~plot_ds["qrange_significant"]
for ax, season in zip(panels.axes[0], change["season"].values):
    plot_hatch(ax, not_significant.sel(season=season), hatch="////", color=(0, 0, 0, 0.3), linewidth=0.3)

panels.fig.legend(
    handles=[Patch(facecolor="white", edgecolor="0.25", hatch="////",
                   label=f"Inside the {sig.N_BOOTSTRAP_MEMBERS}-member hist-nat range (p ≥ {ALPHA})")],
    loc="lower center", frameon=False, handlelength=2.2, handleheight=1.2, bbox_to_anchor=(0.5, 0.005),
)

## 3. Quantile response: Q05, Q50, Q95

$\Delta Q_p$, the final 21 years of each experiment against hist-nat's whole record. The tail dominance $\Delta Q_{05} - \Delta Q_{95}$ is positive where the cold tail warms more than the warm tail (the distribution narrows from below), negative where the warm tail warms more.

In [ ]:
delta_q05_da = quantile_response_da.sel(quantile=0.05, drop=True).rename('delta_q05')
delta_q50_da = quantile_response_da.sel(quantile=0.50, drop=True).rename('delta_q50')
delta_q95_da = quantile_response_da.sel(quantile=0.95, drop=True).rename('delta_q95')
tail_dominance_da = (delta_q05_da - delta_q95_da).rename('tail_dominance')

Multi-model medians of the median warming $\Delta Q_{50}$ and of the tail dominance, hatched where fewer than 80% of the models agree on the sign.

In [ ]:
#(t): Multi-model medians; hatched where fewer than SIGN_THRESHOLD of models agree on the sign
for name, da, cmap in [
    ('Median warming ΔQ50', delta_q50_da, 'RdBu_r'),
    ('Tail dominance ΔQ05 − ΔQ95', tail_dominance_da, 'BrBG'),
]:
    mm.model_agreement_maps(sig.model_agreement(da), title=f'{name}: multi-model median', cbar_label=name, cmap=cmap)

Single-model detail (`PLOT_SEL`):

In [ ]:
quantile_response_plot = quantile_response_da.sel(**PLOT_SEL).assign_coords(
    quantile=["Q05", "Q50", "Q95"]
)

limit = float(
    np.nanquantile(
        np.abs(quantile_response_plot.values),
        0.98,
    )
)

quantile_panels = polar_grid(
    quantile_response_plot,
    row_dim="quantile",
    col_dim="season",
    levels=np.linspace(-limit, limit, 13),
    cmap="RdBu_r",
    title=f"{var.long_name}: response by quantile",
    cbar_label=f"Experiment − hist-nat ({UNITS})",
    tag=False,
)

The tail dominance for one model, dotted where the change in width is not significant: there the two tails have not moved apart by more than natural variability allows.

In [ ]:
tail_limit = float(
    np.nanquantile(
        np.abs(tail_dominance_da.sel(**PLOT_SEL).values),
        0.98,
    )
)

tail_panels = polar_grid(
    tail_dominance_da.sel(**PLOT_SEL),
    col_dim="season",
    levels=np.linspace(-tail_limit, tail_limit, 13),
    cmap="BrBG",
    title="Relative warming of cold and warm tails",
    cbar_label="ΔQ05 − ΔQ95",
    tag=False,
)

In [ ]:
#(t): Hatched where the change in width is not significant (hist-nat bootstrap)
width_not_significant = (qrange_change_ds.sel(**PLOT_SEL)["qrange_pvalue"].notnull()
                         & ~qrange_change_ds.sel(**PLOT_SEL)["qrange_significant"])

for ax, season in zip(
    tail_panels.axes[0],
    tail_dominance_da.sel(**PLOT_SEL)["season"].values,
):
    plot_hatch(
        ax,
        width_not_significant.sel(season=season),
        hatch="....",
        color=(0, 0, 0, 0.25),
        linewidth=0.25,
    )

Both at once: the colour is the tail dominance (brown: the cold tail warms more; green: the warm tail does) and its strength the median warming. The square on the right is the key.

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from plotting_modules import maps


def bivariate_rgba(
    warming,
    tail_dominance,
    warming_min,
    warming_max,
    tail_limit,
):
    """Brown/green hue for tail dominance; saturation for median warming."""
    tail_norm = np.clip(
        (tail_dominance / tail_limit + 1) / 2,
        0,
        1,
    )

    base_rgb = plt.get_cmap("BrBG")(tail_norm)[..., :3]

    warming_norm = np.clip(
        (warming - warming_min)
        / (warming_max - warming_min),
        0,
        1,
    )

    # Keep weak responses visible, but make stronger warming more saturated.
    strength = 0.15 + 0.85 * warming_norm

    rgb = 1 - strength[..., None] * (1 - base_rgb)

    alpha = (
        np.isfinite(warming)
        & np.isfinite(tail_dominance)
    ).astype(float)

    return np.concatenate(
        [rgb, alpha[..., None]],
        axis=-1,
    )


warming_min = float(
    np.nanquantile(delta_q50_da.sel(**PLOT_SEL).values, 0.02)
)
warming_max = float(
    np.nanquantile(delta_q50_da.sel(**PLOT_SEL).values, 0.98)
)

tail_limit = float(
    np.nanquantile(
        np.abs(tail_dominance_da.sel(**PLOT_SEL).values),
        0.98,
    )
)

seasons = delta_q50_da.sel(**PLOT_SEL)["season"].values

fig = plt.figure(figsize=(13, 3.3))

axes = [
    fig.add_subplot(
        1,
        len(seasons) + 1,
        i + 1,
        projection=ccrs.SouthPolarStereo(),
    )
    for i in range(len(seasons))
]

axes = [maps.setup_polar_ax(ax) for ax in axes]

for ax, season in zip(axes, seasons):
    warming = delta_q50_da.sel(**PLOT_SEL).sel(season=season)
    tail = tail_dominance_da.sel(**PLOT_SEL).sel(season=season)

    rgba = bivariate_rgba(
        warming.values,
        tail.values,
        warming_min,
        warming_max,
        tail_limit,
    )

    ax.pcolormesh(
        warming.lon,
        warming.lat,
        rgba,
        transform=ccrs.PlateCarree(),
        shading="auto",
    )

    ax.set_title(str(season))


# ============================================================
# 2-D BIVARIATE LEGEND
# ============================================================

legend_ax = fig.add_subplot(
    1,
    len(seasons) + 1,
    len(seasons) + 1,
)

legend_warming = np.linspace(
    warming_min,
    warming_max,
    150,
)

legend_tail = np.linspace(
    -tail_limit,
    tail_limit,
    150,
)

legend_warming_2d, legend_tail_2d = np.meshgrid(
    legend_warming,
    legend_tail,
)

legend_rgba = bivariate_rgba(
    legend_warming_2d,
    legend_tail_2d,
    warming_min,
    warming_max,
    tail_limit,
)

legend_ax.imshow(
    legend_rgba,
    origin="lower",
    extent=[
        warming_min,
        warming_max,
        -tail_limit,
        tail_limit,
    ],
    aspect="auto",
)

legend_ax.axhline(0, color="0.4", lw=0.7)

legend_ax.set_xlabel("Median warming ΔQ50")
legend_ax.set_ylabel("ΔQ05 − ΔQ95")

legend_ax.text(
    0.5,
    1.03,
    "Cold tail faster",
    ha="center",
    transform=legend_ax.transAxes,
)

legend_ax.text(
    0.5,
    -0.20,
    "Warm tail faster",
    ha="center",
    transform=legend_ax.transAxes,
)

fig.suptitle(
    "Median warming and relative tail response",
    y=1.02,
)

fig.tight_layout()

### Across the distribution

The response at every 5th percentile, Antarctic (area-weighted) mean: the multi-model median (line) and the range across models (shading). A flat line is a pure shift.

In [ ]:
experiments = antarctic_quantile_curve_da['experiment'].values
fig, axes = plt.subplots(1, len(experiments), figsize=(4 * len(experiments), 4), sharey=True, layout='constrained')
axes = np.atleast_1d(axes)

for ax, exp in zip(axes, experiments):
    for i, season in enumerate(antarctic_quantile_curve_da['season'].values):
        da = antarctic_quantile_curve_da.sel(experiment=exp, season=season)
        percentile = da['quantile'] * 100
        ax.fill_between(percentile, da.min('model'), da.max('model'), color=f'C{i}', alpha=0.15, linewidth=0)
        ax.plot(percentile, da.median('model'), color=f'C{i}', marker='o', ms=3, label=str(season))
    ax.axhline(0, color='0.5', lw=0.8)
    ax.set(title=str(exp), xlabel='Percentile')

axes[0].set_ylabel(f'Response, experiment − hist-nat ({UNITS})')
axes[0].legend(title='Season', frameon=False, ncol=2)
fig.suptitle(f'Antarctic {var.long_name.lower()} response across the distribution '
             '(multi-model median; shading = model range)');

## 4. Mean change

Welch t-test and member-block permutation test of the final 21 years against hist-nat's (notebook 03, section 4).

In [ ]:
ttest_stipple_mask = maps.stipple_mask(
    ttest_ds.p,
    threshold=0.05,
)

### Multi-model summary

The median $t$ across models, with IPCC AR6 hatching: `////` where fewer than 66% of the models are significant, `xxxx` where enough are but fewer than 80% agree on the sign. Then the Antarctic-mean $t$ per model, with the share of the area where $p < 0.05$.

In [ ]:
#(t): Median t; significant where p < 0.05
ttest_final = ttest_ds.sel(experiment=FORCED_EXPERIMENTS)
ttest_significant = ttest_final.p < 0.05

mm.model_agreement_maps(
    sig.model_agreement(ttest_final.t, ttest_significant,
                                 sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD),
    title=f'Welch t-test, final {WINDOW} years vs hist-nat: multi-model median',
    cbar_label='t',
)

In [ ]:
#(t): Text = % of the area with p < 0.05
mm.model_heatmap(
    sig.area_mean(ttest_final.t).compute(),
    title=f'Antarctic-mean t (final {WINDOW} years vs hist-nat)',
    cbar_label='t',
    annot=sig.area_mean(ttest_significant.where(ttest_final.t.notnull())).compute(),
)

The member-block test, which respects the serial correlation of each member's years:

In [ ]:
#(t): Median change; significant where the member-block test gives p < 0.05
block_final = mean_block_ds.sel(experiment=FORCED_EXPERIMENTS)
mm.model_agreement_maps(
    sig.model_agreement(block_final.mean_change, block_final.pvalue < 0.05,
                                 sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD),
    title=f'Mean change, final {WINDOW} years vs hist-nat (member-block test): multi-model median',
    cbar_label=f'Mean change ({UNITS})',
)

### Single-model detail: median change, hatched where the t-test gives p < 0.05

In [ ]:
panels = maps.polar_grid(
    final_change_quantile.sel(experiment=FORCED_EXPERIMENTS, model=[PLOT_SEL['model']])[VARIABLE],
     col_dim='experiment',
    row_dim='model',
    sel={'season': 'DJF', 'quantile': 0.5},
    levels=np.arange(-5, 5.5, .5) / 2,
    cmap='RdBu_r',
    cbar_label='Anomaly ' + f'({UNITS})',
)

maps.map_panels(panels, ttest_stipple_mask.sel(experiment=FORCED_EXPERIMENTS, model=[PLOT_SEL['model']]), maps.plot_hatch,
           row_dim='model', col_dim='experiment', sel={'season': 'DJF'});

## 5. Signal to noise

$\mathrm{S/N} = S / N$, the forced difference from hist-nat over the pooled rolling noise (notebook 03, section 5). White is $|\mathrm{S/N}| < 2$.

### Multi-model summary

In [ ]:
#(t): S/N at the year shown in the maps; a model counts as significant where |S/N| >= SN_THRESHOLD
#(c): The centre of the final WINDOW years, the last year with a complete noise window, so S/N covers the same years
#(c): as the t-test, the member-block test and the width test
SN_YEAR_INDEX = LAST_FULL_YEAR
SN_THRESHOLD = 2   # the white band of sn_cmap

sn_final = lesfmip_seasonal_sn_da[VARIABLE].isel(year=SN_YEAR_INDEX).sel(experiment=FORCED_EXPERIMENTS)
sn_emerged = np.abs(sn_final) >= SN_THRESHOLD

mm.model_agreement_maps(
    sig.model_agreement(sn_final, sn_emerged,
                                 sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD),
    title=f'S/N ratio ({int(sn_final.year)}): multi-model median',
    cbar_label='S/N ratio',
    levels=sn_bounds,
    cmap=sn_cmap,
)

In [ ]:
#(t): Text = % of the area with |S/N| >= SN_THRESHOLD
mm.model_heatmap(
    sig.area_mean(sn_final).compute(),
    title=f'Antarctic-mean S/N ratio ({int(sn_final.year)})',
    cbar_label='S/N ratio',
    annot=sig.area_mean(sn_emerged.where(sn_final.notnull())).compute(),
)

### Single-model detail

S/N for `PLOT_SEL['model']` in every season and experiment, then every model in DJF, hatched where the t-test gives $p < 0.05$: where the t-test finds a change that has not emerged from the noise, the hatching sits on white.

In [ ]:
fig = maps.polar_grid(
    lesfmip_seasonal_sn_da.isel(year=SN_YEAR_INDEX).sel(experiment=MAIN_EXPERIMENTS)[VARIABLE],
    row_dim='experiment',
    col_dim='season',
    # sel={'season': 'DJF'},#, 'quantile': 0.5},
    sel={'model': PLOT_SEL['model']},
    levels=sn_bounds, 
    cmap=sn_cmap,
    cbar_label='S/N Ratio',
    # title=True
)

In [ ]:
panels = maps.polar_grid(
    lesfmip_seasonal_sn_da.isel(year=SN_YEAR_INDEX).sel(experiment=FORCED_EXPERIMENTS)[VARIABLE],
    col_dim='experiment',
    row_dim='model',
    sel={'season': 'DJF'},#, 'quantile': 0.5},
    levels=sn_bounds, 
    cmap=sn_cmap,
    cbar_label='S/N Ratio',
    # title=True
)
maps.map_panels(panels, ttest_stipple_mask.sel(experiment=FORCED_EXPERIMENTS), maps.plot_hatch,
           row_dim='model', col_dim='experiment', sel={'season': 'DJF'});

## 6. Share of the historical change from each forcing

Each single forcing's $\Delta Q_{50}$ as a percentage of historical's, at the last complete year:

$$
f_e = 100 \times \frac{\Delta Q_{50}^{\,e}}{\Delta Q_{50}^{\,\mathrm{historical}}}.
$$

It blows up where historical's change is small, so read it only where historical has changed clearly. Whether the parts add up to historical is section 7, which measures each experiment from its own baseline: measured from hist-nat, as here, a sum of four forcings would subtract hist-nat's volcanic cooling four times.

In [ ]:
ALL_BUT_HISTORICAL = ['hist-GHG', 'hist-aer', 'hist-nat', 'hist-totalO3']

In [ ]:
fraction_of_historical = (
    lesfmip_change_quantile_da.sel(quantile=0.5, experiment=ALL_BUT_HISTORICAL)
    /lesfmip_change_quantile_da.sel(quantile=0.5).sel(experiment='historical')
)[VARIABLE]

### Multi-model summary

In [ ]:
#(t): Percent of the historical ΔQ50 from each single forcing, final valid year; hatching is sign agreement only
mm.model_agreement_maps(
    sig.model_agreement(100 * fraction_of_historical.isel(year=LAST_FULL_YEAR), sign_threshold=SIGN_THRESHOLD),
    title='Percent of historical ΔQ50: multi-model median',
    cbar_label='Percent of historical',
    levels=frac_bounds * 100,
    cmap=my_cmap,
)

### Per model, DJF

In [ ]:
fig = maps.polar_grid(
    100*fraction_of_historical.isel(year=LAST_FULL_YEAR).sel(experiment=['hist-nat', 'hist-aer', 'hist-totalO3', 'hist-GHG']),
    row_dim='experiment',
    col_dim='model',
    sel={'season': 'DJF'},#, 'quantile': 0.5},
    levels=frac_bounds*100,#np.arange(-2, 2.2, .2),
    cmap=my_cmap,
    cbar_label='Percent of historical',
    # title=True
)

## 7. Additivity: does historical equal the sum of the single forcings?

Single-forcing experiments contain *only* their forcing, so each one's response is best measured from its own pre-industrial baseline (1850–1900), not from hist-nat's full record. Measuring everything from hist-nat's full-record mean subtracts hist-nat's average volcanic cooling from every single forcing, so a sum of four forcings would subtract it four times while historical subtracts it once.

The **residual**, historical − (nat + GHG + aer + O3), holds everything the parts leave out: land use (hist-lu, which should be very small over Antarctica), any other forcing, and non-additive interaction between forcings. Fractions of historical are only given where |historical| ≥ 0.25 °C; elsewhere they blow up.

In [ ]:
additivity_ds = rc.additivity(own_changes_da)

One model in DJF: each part, their sum, historical, and the residual (historical minus the sum).

In [ ]:
#(t): One model, every term (DJF): the parts, their sum, historical and the residual
additivity_model = additivity_ds.response.sel(model=PLOT_SEL['model'], season='DJF')
panels = maps.polar_grid(
    additivity_model, col_dim='term', levels=rcp.symmetric_levels(additivity_model), cmap='RdBu_r',
    title=f"{PLOT_SEL['model']}, DJF: final {WINDOW} years minus 1850-1900", cbar_label=f'Change ({UNITS})', tag=False,
)

The residual for every model and season. Land use, other forcings and any interaction between forcings are in it.

In [ ]:
#(t): The residual (historical - sum of parts) for every model and season
residual = additivity_ds.response.sel(term='residual')
panels = maps.polar_grid(
    residual, row_dim='model', col_dim='season', levels=rcp.symmetric_levels(residual), cmap='RdBu_r',
    title='Historical minus the sum of the single forcings', cbar_label=f'Residual ({UNITS})', tag=False,
)

Antarctic means (south of 60°S) of every term, DJF.

In [ ]:
#(t): Antarctic mean (south of 60°S) of every term, per model, DJF
(sig.area_mean(additivity_ds.response.sel(lat=slice(None, -60), season='DJF'))
 .to_series().unstack('term')[list(additivity_ds.term.values)].round(2))

## 8. Mean and variability change

The end point of the analysis. For each forcing:
1. **the temperature response**, and whether it is significant;
2. **whether the variability has changed**: a wider or narrower distribution, and whether the cold or the warm tail moves faster;
3. **where both the mean and the variability have changed**.

This section combines results from notebook 03:

| Quantity | Where it comes from | Test |
|---|---|---|
| Mean response | `rc.mean_response`: final `WINDOW` (21) years of each experiment minus hist-nat over the same years, members pooled | member-block permutation test (`mean_block_ds`, p < `alpha`) and S/N against hist-nat (`sn_final`, \|S/N\| ≥ `SN_THRESHOLD`) |
| Width change, Q95 − Q05 | `qrange_change_ds['qrange_change']`: the final `WINDOW` years vs hist-nat's full record, on `qrange_tree` | hist-nat bootstrap (`qrange_pvalue` < `alpha`) |
| Tail changes | `rc.tail_changes(qrange_tree)`: Δ(Q95 − Q50) and Δ(Q50 − Q05), against the same full-record hist-nat | the width test, split between the tails; bootstrap at a point (part 2) |

**The tails.** ΔQ95 − ΔQ05 *is* the change in width, so "does the warm tail warm faster than the cold tail" is the width question, and ΔQ05 − ΔQ95 is just minus the width change. What the width does not say is *which* tail did the widening. Measuring each tail from the median answers that: the warm tail Δ(Q95 − Q50) and the cold tail Δ(Q50 − Q05) add up to the width change, and their difference (the tail asymmetry) says which one stretched more.

**Which test for the mean?** The member-block test (see the T-test section) asks whether the ensemble mean has shifted, respecting that a member's years are correlated; with ~50 members it can detect small shifts. S/N asks the stricter question of whether the shift stands out from year-to-year noise, i.e. whether a single realisation, like the real world, would show it. By default the mean change counts as *robust* only when both agree; `MEAN_TEST` switches to either alone, and `mean_pvalue_da = ttest_ds.p` uses the pooled Welch t-test instead.

**Credibility.** A model's response is only as trustworthy as its simulation of the observed signal and noise. The ERA5 evaluation (notebook 02: the scorecard, and the maps in its section 7) shows where each model's trend, variability and width are consistent with ERA5.

Calculations live in `extant/response_change.py` (`rc`) and figures in `extant/plots/response_change.py` (`rcp`).

In [ ]:
#(t): Settings
CHANGE_YEARS = WINDOW                     # final years compared with hist-nat, as in the tests
MEAN_TEST = 'robust'                      # 'significant' (mean test), 'emerged' (S/N) or 'robust' (both)
mean_pvalue_da = mean_block_ds.pvalue     # the member-block test; ttest_ds.p for the pooled Welch t-test
CHANGE_MODEL = PLOT_SEL['model']          # model shown in the single-model figures
CHANGE_SEASON = 'DJF'                     # season shown in the single-season figures

In [ ]:
#(t): Every change and its test in one Dataset on (model, experiment, season, lat, lon)
change_ds = rc.change_summary(
    mean_response_da, mean_pvalue_da, sn_final, qrange_change_ds, tails_ds,
    alpha=ALPHA, sn_threshold=SN_THRESHOLD,
)
change_ds

## 1. Maps: where the mean and the width have changed

Colour is the mean response, drawn only where it is robust (white elsewhere). Hatching marks a significant change in the width of the distribution: `////` wider, `\\\\` narrower. So a coloured, hatched region is where **both the mean and the variability have changed**; a white, hatched region is where only the variability has.

In [ ]:
#(t): One model: experiments x seasons
panels = rcp.joint_change_maps(change_ds, CHANGE_MODEL, mean_test=MEAN_TEST, ice=ice_climatology)

In [ ]:
#(t): Every model: how many show a robust mean change together with a significant widening, or narrowing
joint_counts = rc.joint_change_counts(change_ds, mean_test=MEAN_TEST)
for season in ('DJF', 'JJA'):
    panels = rcp.joint_change_count_maps(joint_counts, season, n_models=change_ds.sizes['model'])

## 2. A closer look: how the distribution shifts and widens at one point

At a single grid point, the final years of each experiment (members pooled) against hist-nat:
- **a)** the two distributions, with arrows from each hist-nat quantile (Q05, Q50, Q95) to the experiment's. Equal arrows are a pure shift. A longer Q95 arrow means the warm tail warms faster and the distribution widens upwards; a longer Q05 arrow, the cold tail.
- **b)** the change in every quantile, for every experiment, with a bootstrap (whole members) 5–95% range. A flat curve is a pure shift; a rising curve, the warm tail warms faster; a falling curve, the cold tail does.

By default the point is where the mean and the width both changed and the width changed most (south of 60°S). Set `change_point` to look anywhere else.

In [ ]:
#(t): The point: where the mean and the width both changed, and the width changed most
CHANGE_EXPERIMENT = 'historical'
change_point = rc.strongest_joint_change(change_ds, CHANGE_MODEL, CHANGE_EXPERIMENT, CHANGE_SEASON,
                                         mean_test=MEAN_TEST)
# change_point = dict(lat=-75.0, lon=-180.0)   # or choose one
change_point

In [ ]:
#(t): Final years at that point for hist-nat and each experiment, and the change in every quantile
point_experiments = [e for e in MAIN_EXPERIMENTS if e in lesfmip_season_tree[CHANGE_MODEL].children]
point_samples = rc.final_years(lesfmip_season_tree, CHANGE_MODEL, point_experiments, change_point, CHANGE_SEASON,
                               years=CHANGE_YEARS, variable=VARIABLE)
point_shifts = {e: rc.quantile_shift(point_samples[e], point_samples['hist-nat'])
                for e in point_experiments if e != 'hist-nat'}
panels = rcp.shift_and_widen(
    point_samples, point_shifts, CHANGE_EXPERIMENT,
    test=change_ds.sel(model=CHANGE_MODEL, experiment=CHANGE_EXPERIMENT, season=CHANGE_SEASON).sel(change_point),
    title=f'How the distribution changes: {CHANGE_MODEL}, {CHANGE_SEASON} at {formatting.format_latlon(change_point)}',
)

## 3. Every model and forcing: mean change against variability change

Area-weighted means south of 60°S, one point per model (marker) and experiment (colour), one panel per season:
1. the mean response against the change in width;
2. the cold-tail change against the warm-tail change: above the dashed 1:1 line the warm tail stretched more, below it the cold tail;
3. the mean response against the tail asymmetry.

In [ ]:
#(t): Regional means south of 60°S, per model, experiment and season
change_regional = rc.regional_mean(change_ds, lat_max=-60).compute()
panels = rcp.response_scatter(change_regional, 'mean_change', 'width_change')

In [ ]:
#(t): Which tail did the widening: cold-tail change against warm-tail change
panels = rcp.response_scatter(change_regional, 'lower_tail_change', 'upper_tail_change', diagonal=True)

In [ ]:
#(t): Mean response against the tail asymmetry (warm-tail minus cold-tail change)
panels = rcp.response_scatter(change_regional, 'mean_change', 'tail_asymmetry')

In [ ]:
#(t): % of the region (south of 60°S) where neither, only the mean, only the width, or both changed
class_columns = [f'fraction_{name.replace(" ", "_")}' for name in rc.CHANGE_CLASSES.values()]
(100 * change_regional[class_columns].sel(season=CHANGE_SEASON)).to_dataframe().round(0)

**Reading these figures**
- A forcing that only **shifts** the distribution lies along the x axis of the scatter, and its quantile curve (part 2 b) is flat.
- A forcing that also **widens** it sits above the x axis, and its quantile curve rises (warm tail faster) or falls (cold tail faster). Which tail did the stretching is read off the tail scatter.
- Regions coloured *and* hatched on the maps are where both the mean and the variability changed. Check them against the ERA5 evaluation maps before interpreting: a width change is only credible where the model's present-day variability is consistent with ERA5 (the std, skewness and kurtosis maps and their field tests in the ERA5 evaluation).

## 9. Zonal structure and the sea-ice edge

After Bracegirdle et al. (2024, *npj Clim. Atmos. Sci.* 7, 276; [doi:10.1038/s41612-024-00822-y](https://doi.org/10.1038/s41612-024-00822-y)), Figs 1–3, 6 and 8, which compare the change in extreme seasons with the change in the background climate, latitude by latitude, and relate the difference to the retreating sea ice. Their methods differ from ours, so these figures keep their layout but show this project's changes:

| Column | Here | The paper |
|---|---|---|
| Mean | $\Delta$ mean, the mean response (final years against hist-nat) | the background climate (smoothed ensemble mean) |
| Low extremes | $\Delta(Q_{05} - Q_{50})$ of the internal variability: > 0 where the low extremes move up towards the middle (for temperature, cold extremes warming faster than the median) | p10, minus the background |
| High extremes | $\Delta(Q_{95} - Q_{50})$: > 0 where the high extremes move away from the middle | p90, minus the background |
| Width | $\Delta(Q_{95} - Q_{05})$ = high − low | p90 − p10 |

Other differences: **every model is shown on its own** (the paper averages five), the changes are each forcing's against hist-nat (the paper compares 2080 with 1960 in one scenario), and the zonal means are tested (notebook 03, section 8): dots mark a significant change in the mean (member-block test) or the width (hist-nat bootstrap); the grey band is what hist-nat's own variability gives the width. The dashed lines are the sea-ice edge (equivalent latitude, notebook 01 section 5) in each experiment's final years.

The paper's key result for temperature: in winter, over the retreating sea ice, the cold extremes warm much more than the warm extremes, so the distribution narrows. Here that would show as a positive low-extreme change and a negative width change just poleward of the ice edge, in the experiments where the edge has moved.

In [ ]:
#(t): The four columns, zonally averaged, with the zonal tests
zonal_ds = zonal.zonal_summary(change_ds, zonal_mean_ds, zonal_width_ds)
zonal_ds

### The change at each latitude, every model

One row per model, the experiments as lines. The mean has its own scale; the extremes and the width share one, so the width column is the high column minus the low.

In [ ]:
for season in ('DJF', 'JJA'):
    fig, axes = zp.zonal_change_grid(zonal_ds, season, edges=ice_edges, units=UNITS)

The same, the other way round: one row per experiment and the models as lines, to see how far the models agree on each forcing.

In [ ]:
fig, axes = zp.zonal_change_grid(zonal_ds, CHANGE_SEASON, rows='experiment', lines='model', edges=ice_edges,
                                 units=UNITS)

### Where the seasons vary most

hist-nat's own width $Q_{95} - Q_{05}$ (full record), zonally averaged, with each model's hist-nat ice edge. The paper finds the year-to-year variability of winter temperature largest just poleward of the ice edge, where a season can be over ice or open water; a model whose peak sits elsewhere, or whose edge is far from the others', will put its changes in variability at other latitudes too.

In [ ]:
fig, axes = zp.zonal_reference_width(zonal_ds, edges=ice_edges, units=UNITS)

### The maps behind the zonal means

One model and experiment, the same four columns as maps (the paper's Figs 1 and 3), with the 15% ice edge of the experiment (solid) and of hist-nat (dashed). Dotted where the mean or the width change is not significant.

In [ ]:
for experiment in ('historical', 'hist-GHG'):
    panels = rcp.extreme_change_maps(change_ds, CHANGE_MODEL, experiment, mean_test=MEAN_TEST, ice=ice_climatology,
                                     units=UNITS)

### The sea ice itself (JASMIN)

Each experiment's extent through time (ensemble mean and the 5–95% of members), and the equivalent latitude of each model's edge in the final years.

In [ ]:
fig, axes = sip.extent_timeseries(sea_ice.extent_summary(ice_extents))
ice_edges.to_dataframe().unstack('season')['edge_latitude'].round(1)